# STRAT-004: Entry & Exit Strategy Matrix

## Entry Variants:
- **Entry 1 (Original)**: Fixed thresholds only (MVRV < 1, SOPR < 1, etc.)
- **Entry 2 (Z-Confirmed)**: Fixed thresholds + Z-score < -1.0 (unusually distressed)

## Exit Variants:
- **Baseline**: 10% Trailing Stop
- **Variant A**: Simple Mirror (all 5 flip bullish)
- **Variant B**: 6/8 Z-Score Confluence (euphoria)
- **Variant C**: LTH Distribution (MVRV > 2 + LTH-SOPR > 1.5)
- **Hybrid**: Trail + tighten when signal fires

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (16, 8)

print("STRAT-004: Entry & Exit Strategy Matrix 🎯")

## 1. Load All Required Metrics

In [ ]:
BRK_DIR = Path("../data/brk/daily")
GN_DIR = Path("../data/glassnode/daily")

def load_metric(filepath, metric_name):
    """Load metric handling both BRK and Glassnode formats."""
    df = pd.read_parquet(filepath)
    if 'time' in df.columns:
        df = df.set_index('time')
    if 'value' in df.columns:
        df = df.rename(columns={'value': metric_name})
    return df[[metric_name]]

# === ENTRY METRICS ===
price = load_metric(BRK_DIR / "price.parquet", "price")
mvrv_sth = load_metric(BRK_DIR / "mvrv_sth.parquet", "mvrv_sth")
sopr_sth = load_metric(BRK_DIR / "sopr_sth.parquet", "sopr_sth")
realized_profit = load_metric(BRK_DIR / "realized_profit.parquet", "realized_profit")
realized_loss = load_metric(BRK_DIR / "realized_loss.parquet", "realized_loss")
funding_rate = load_metric(GN_DIR / "funding_rate.parquet", "funding_rate")
liq_long = load_metric(GN_DIR / "liquidations_long.parquet", "liq_long")
liq_short = load_metric(GN_DIR / "liquidations_short.parquet", "liq_short")

# === EXIT METRICS ===
mvrv = load_metric(BRK_DIR / "mvrv.parquet", "mvrv")
sopr = load_metric(BRK_DIR / "sopr.parquet", "sopr")
sopr_lth = load_metric(BRK_DIR / "sopr_lth.parquet", "sopr_lth")
price_200sma = load_metric(BRK_DIR / "price_200d_sma.parquet", "price_200sma")
puell = load_metric(BRK_DIR / "puell_multiple.parquet", "puell")
sell_side_risk = load_metric(BRK_DIR / "sell_side_risk.parquet", "sell_side_risk")

print("All metrics loaded ✓")

In [ ]:
# Combine all metrics
df = price.copy()
for m in [mvrv_sth, sopr_sth, realized_profit, realized_loss, 
          funding_rate, liq_long, liq_short,
          mvrv, sopr, sopr_lth, price_200sma, puell, sell_side_risk]:
    df = df.join(m, how='left')

# Remove duplicates
if df.index.duplicated().any():
    print(f"Removing {df.index.duplicated().sum()} duplicate indices")
    df = df[~df.index.duplicated(keep='last')]

df = df.sort_index()

# Calculate derived metrics
df['rplr'] = df['realized_profit'] / df['realized_loss'].replace(0, np.nan)
df['liq_ratio'] = df['liq_long'] / df['liq_short'].replace(0, np.nan)
df['mayer_multiple'] = df['price'] / df['price_200sma']

print(f"Combined dataset: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Calculate Z-scores (rolling 1-year window)
z_window = 365

def calc_zscore(series, window=365):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std

# Exit Z-scores (for euphoria detection)
df['mvrv_z'] = calc_zscore(df['mvrv'], z_window)
df['mvrv_sth_z'] = calc_zscore(df['mvrv_sth'], z_window)
df['sopr_z'] = calc_zscore(df['sopr'], z_window)
df['sopr_sth_z'] = calc_zscore(df['sopr_sth'], z_window)
df['mayer_z'] = calc_zscore(df['mayer_multiple'], z_window)
df['puell_z'] = calc_zscore(df['puell'], z_window)
df['ssr_z'] = calc_zscore(df['sell_side_risk'], z_window)
df['funding_z'] = calc_zscore(df['funding_rate'], z_window)

# Entry Z-scores (for distress confirmation)
df['rplr_z'] = calc_zscore(df['rplr'], z_window)
df['liq_ratio_z'] = calc_zscore(df['liq_ratio'], z_window)

print("Z-scores calculated ✓")

## 2. Create Entry Signals

### Entry 1 (Original): Fixed Thresholds Only
- Economic meaning: underwater, selling at loss, etc.

### Entry 2 (Z-Confirmed): Fixed + Z < -1.0
- Not just underwater, but UNUSUALLY underwater vs recent history

In [ ]:
# Focus on derivatives era
df_full = df[df.index >= '2020-02-01'].copy()

# ===== ENTRY 1: FIXED THRESHOLDS ONLY =====
df_full['cond_mvrv_fixed'] = df_full['mvrv_sth'] < 1.0
df_full['cond_sopr_fixed'] = df_full['sopr_sth'] < 1.0
df_full['cond_rplr_fixed'] = df_full['rplr'] < 1.0
df_full['cond_funding_fixed'] = df_full['funding_rate'] <= 0
df_full['cond_liq_fixed'] = df_full['liq_ratio'] > 1.0

df_full['signal_fixed'] = (
    df_full['cond_mvrv_fixed'] & 
    df_full['cond_sopr_fixed'] & 
    df_full['cond_rplr_fixed'] &
    df_full['cond_funding_fixed'] &
    df_full['cond_liq_fixed']
)

# ===== ENTRY 2: FIXED + Z-SCORE CONFIRMED =====
# Fixed thresholds (same as above)
# PLUS: Z-score < -1.0 for on-chain metrics (unusually distressed)
df_full['cond_mvrv_z'] = (df_full['mvrv_sth'] < 1.0) & (df_full['mvrv_sth_z'] < -1.0)
df_full['cond_sopr_z'] = (df_full['sopr_sth'] < 1.0) & (df_full['sopr_sth_z'] < -1.0)
df_full['cond_rplr_z'] = (df_full['rplr'] < 1.0) & (df_full['rplr_z'] < -1.0)
# Derivatives: keep fixed (Z less meaningful for funding/liquidations)
df_full['cond_funding_z'] = df_full['funding_rate'] <= 0
df_full['cond_liq_z'] = df_full['liq_ratio'] > 1.0

df_full['signal_z_confirmed'] = (
    df_full['cond_mvrv_z'] & 
    df_full['cond_sopr_z'] & 
    df_full['cond_rplr_z'] &
    df_full['cond_funding_z'] &
    df_full['cond_liq_z']
)

# ===== ENTRY 3: FIXED + PARTIAL Z (at least 2/3 on-chain Z-confirmed) =====
df_full['z_confirm_count'] = (
    ((df_full['mvrv_sth'] < 1.0) & (df_full['mvrv_sth_z'] < -1.0)).astype(int) +
    ((df_full['sopr_sth'] < 1.0) & (df_full['sopr_sth_z'] < -1.0)).astype(int) +
    ((df_full['rplr'] < 1.0) & (df_full['rplr_z'] < -1.0)).astype(int)
)

df_full['signal_partial_z'] = (
    df_full['cond_mvrv_fixed'] &  # Still need all fixed conditions
    df_full['cond_sopr_fixed'] & 
    df_full['cond_rplr_fixed'] &
    df_full['cond_funding_fixed'] &
    df_full['cond_liq_fixed'] &
    (df_full['z_confirm_count'] >= 2)  # At least 2/3 Z-confirmed
)

# Entry = first day signal turns on
df_full['entry_fixed'] = df_full['signal_fixed'] & ~df_full['signal_fixed'].shift(1).fillna(False)
df_full['entry_z_confirmed'] = df_full['signal_z_confirmed'] & ~df_full['signal_z_confirmed'].shift(1).fillna(False)
df_full['entry_partial_z'] = df_full['signal_partial_z'] & ~df_full['signal_partial_z'].shift(1).fillna(False)

# Drop rows with missing key data
required = ['price', 'mvrv_sth', 'sopr_sth', 'rplr', 'funding_rate', 'liq_ratio',
            'mvrv_sth_z', 'sopr_sth_z', 'rplr_z']
df_full = df_full.dropna(subset=required)

print(f"Dataset: {len(df_full)} rows")
print(f"\nEntry Signal Comparison:")
print(f"  Entry 1 (Fixed only):      {df_full['entry_fixed'].sum()} entries")
print(f"  Entry 2 (Full Z-Confirmed): {df_full['entry_z_confirmed'].sum()} entries")
print(f"  Entry 3 (Partial Z 2/3):   {df_full['entry_partial_z'].sum()} entries")

In [ ]:
# Show entry dates for each variant
print("\n" + "="*80)
print("ENTRY SIGNAL COMPARISON")
print("="*80)

print("\nEntry 1 (Fixed Thresholds Only):")
print("-" * 60)
for d in df_full[df_full['entry_fixed']].index:
    row = df_full.loc[d]
    z_count = row['z_confirm_count']
    print(f"  {d.date()}: ${row['price']:>8,.0f}  MVRV:{row['mvrv_sth']:.3f} (Z:{row['mvrv_sth_z']:+.1f})  "
          f"SOPR:{row['sopr_sth']:.3f} (Z:{row['sopr_sth_z']:+.1f})  Z-confirms:{int(z_count)}/3")

print("\nEntry 2 (Full Z-Confirmed - all 3 on-chain Z < -1):")
print("-" * 60)
z_confirmed_dates = df_full[df_full['entry_z_confirmed']].index
if len(z_confirmed_dates) == 0:
    print("  No entries (too restrictive!)")
else:
    for d in z_confirmed_dates:
        row = df_full.loc[d]
        print(f"  {d.date()}: ${row['price']:>8,.0f}  MVRV:{row['mvrv_sth']:.3f} (Z:{row['mvrv_sth_z']:+.1f})  "
              f"SOPR:{row['sopr_sth']:.3f} (Z:{row['sopr_sth_z']:+.1f})")

print("\nEntry 3 (Partial Z - at least 2/3 on-chain Z < -1):")
print("-" * 60)
for d in df_full[df_full['entry_partial_z']].index:
    row = df_full.loc[d]
    z_count = row['z_confirm_count']
    print(f"  {d.date()}: ${row['price']:>8,.0f}  MVRV:{row['mvrv_sth']:.3f} (Z:{row['mvrv_sth_z']:+.1f})  "
          f"SOPR:{row['sopr_sth']:.3f} (Z:{row['sopr_sth_z']:+.1f})  Z-confirms:{int(z_count)}/3")

## 3. Create Exit Signals

In [ ]:
# === EXIT A: Simple Mirror (all 5 flip bullish) ===
df_full['exit_a'] = (
    (df_full['mvrv_sth'] > 1.0) &      # STH profitable
    (df_full['sopr_sth'] > 1.0) &      # STH selling at profit
    (df_full['rplr'] > 1.0) &          # More profits than losses
    (df_full['funding_rate'] > 0) &    # Bullish funding
    (df_full['liq_ratio'] < 1.0)       # More short liquidations
)

# === EXIT B: 6/8 Z-Score Confluence ===
df_full['z_count_exit'] = (
    (df_full['mvrv_z'] > 1.5).astype(int) +
    (df_full['mvrv_sth_z'] > 1.25).astype(int) +
    (df_full['sopr_z'] > 1.5).astype(int) +
    (df_full['sopr_sth_z'] > 1.0).astype(int) +
    (df_full['mayer_z'] > 1.0).astype(int) +
    (df_full['puell_z'] > 1.5).astype(int) +
    (df_full['ssr_z'] > 1.5).astype(int) +
    (df_full['funding_z'] > 1.5).astype(int)
)
df_full['exit_b'] = df_full['z_count_exit'] >= 6

# === EXIT C: LTH Distribution ===
df_full['exit_c'] = (
    (df_full['mvrv'] > 2.0) &
    (df_full['sopr_lth'] > 1.5)
)

print("Exit signals created:")
print(f"  Exit A (Simple Mirror):   {df_full['exit_a'].sum()} days active ({df_full['exit_a'].mean()*100:.1f}%)")
print(f"  Exit B (6/8 Z-Score):     {df_full['exit_b'].sum()} days active ({df_full['exit_b'].mean()*100:.1f}%)")
print(f"  Exit C (LTH Distribution): {df_full['exit_c'].sum()} days active ({df_full['exit_c'].mean()*100:.1f}%)")

## 4. Backtest Engine

In [ ]:
def backtest_trailing_stop(df, entry_col, trail_pct=0.10, initial_stop=0.15,
                           initial_capital=100000, fee=0.001):
    """Baseline: Trailing stop exit."""
    price_arr = df['price'].values.astype(np.float64)
    entries = df[entry_col].values
    dates = df.index
    entry_indices = np.where(entries)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        entry_price = price_arr[entry_idx]
        peak = entry_price
        stop_price = entry_price * (1 - initial_stop)
        
        exit_idx = len(price_arr) - 1
        exit_reason = 'hold'
        
        for j in range(entry_idx + 1, len(price_arr)):
            price = price_arr[j]
            if price > peak:
                peak = price
                if peak > entry_price:
                    stop_price = max(stop_price, peak * (1 - trail_pct))
            
            if price <= stop_price:
                exit_idx, exit_reason = j, 'trail_stop'
                break
        
        exit_price = price_arr[exit_idx]
        peak_price = max(price_arr[entry_idx:exit_idx+1])
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fee)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'peak_price': peak_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days,
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def backtest_onchain_exit(df, entry_col, exit_col, stop_loss=0.15, 
                          max_hold=365, initial_capital=100000, fee=0.001):
    """Exit when on-chain signal fires OR stop loss."""
    price_arr = df['price'].values.astype(np.float64)
    exit_arr = df[exit_col].values.astype(bool)
    entries = df[entry_col].values
    dates = df.index
    entry_indices = np.where(entries)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        entry_price = price_arr[entry_idx]
        stop_price = entry_price * (1 - stop_loss)
        peak_price = entry_price
        
        exit_idx = None
        exit_reason = None
        
        for j in range(entry_idx + 1, min(entry_idx + max_hold, len(price_arr))):
            price = price_arr[j]
            if price > peak_price:
                peak_price = price
            
            if price <= stop_price:
                exit_idx, exit_reason = j, 'stop_loss'
                break
            elif exit_arr[j]:
                exit_idx, exit_reason = j, 'signal'
                break
        
        if exit_idx is None:
            exit_idx = min(entry_idx + max_hold, len(price_arr) - 1)
            exit_reason = 'max_hold' if entry_idx + max_hold < len(price_arr) else 'end_data'
        
        exit_price = price_arr[exit_idx]
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fee)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'peak_price': peak_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days,
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def backtest_hybrid_exit(df, entry_col, exit_col, trail_pct=0.10, initial_stop=0.15,
                         tight_trail=0.05, initial_capital=100000, fee=0.001):
    """Hybrid: Trail normally, tighten when signal fires."""
    price_arr = df['price'].values.astype(np.float64)
    exit_arr = df[exit_col].values.astype(bool)
    entries = df[entry_col].values
    dates = df.index
    entry_indices = np.where(entries)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        entry_price = price_arr[entry_idx]
        peak = entry_price
        stop_price = entry_price * (1 - initial_stop)
        signal_triggered = False
        
        exit_idx = len(price_arr) - 1
        exit_reason = 'hold'
        
        for j in range(entry_idx + 1, len(price_arr)):
            price = price_arr[j]
            
            if price > peak:
                peak = price
            
            if not signal_triggered and exit_arr[j]:
                signal_triggered = True
                stop_price = max(stop_price, peak * (1 - tight_trail))
            elif signal_triggered:
                if peak > entry_price:
                    stop_price = max(stop_price, peak * (1 - tight_trail))
            else:
                if peak > entry_price:
                    stop_price = max(stop_price, peak * (1 - trail_pct))
            
            if price <= stop_price:
                exit_idx = j
                exit_reason = 'tight_trail' if signal_triggered else 'trail_stop'
                break
        
        exit_price = price_arr[exit_idx]
        peak_price = max(price_arr[entry_idx:exit_idx+1])
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fee)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'peak_price': peak_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days,
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def calc_metrics(trades_df, initial_capital=100000):
    """Calculate performance metrics."""
    if len(trades_df) == 0:
        return None
    
    final_equity = trades_df['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    
    start_date = trades_df['entry_date'].iloc[0]
    end_date = trades_df['exit_date'].iloc[-1]
    years = (end_date - start_date).days / 365.25
    
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    win_rate = (trades_df['net_return'] > 0).mean()
    
    returns = trades_df['net_return'].values
    if returns.std() > 0 and years > 0:
        trades_per_year = len(trades_df) / years
        sharpe = (returns.mean() / returns.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = 0
    
    equity = [initial_capital] + list(trades_df['equity'])
    peak = equity[0]
    max_dd = 0
    for eq in equity:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'sharpe': sharpe,
        'win_rate': win_rate,
        'max_drawdown': max_dd,
        'n_trades': len(trades_df),
        'avg_hold': trades_df['days_held'].mean(),
        'final_equity': final_equity,
    }

print("Backtest engine ready ✓")

## 5. Run Full Matrix Test

In [ ]:
# Define all entry/exit combinations
entry_variants = [
    ('Entry 1: Fixed', 'entry_fixed'),
    ('Entry 2: Full Z', 'entry_z_confirmed'),
    ('Entry 3: Partial Z', 'entry_partial_z'),
]

exit_variants = [
    ('10% Trail', 'trailing', {}),
    ('Exit A: Mirror', 'onchain', {'exit_col': 'exit_a'}),
    ('Exit B: 6/8 Z', 'onchain', {'exit_col': 'exit_b'}),
    ('Exit C: LTH', 'onchain', {'exit_col': 'exit_c'}),
    ('Hybrid C', 'hybrid', {'exit_col': 'exit_c'}),
]

# Buy & Hold benchmark
bh_return = (df_full['price'].iloc[-1] / df_full['price'].iloc[0] - 1)

print("\n" + "=" * 110)
print("FULL STRATEGY MATRIX: ENTRY x EXIT")
print("=" * 110)
print(f"\nBuy & Hold Return: {bh_return:+.1%}")
print(f"Period: {df_full.index.min().date()} to {df_full.index.max().date()}")

# Results storage
all_results = []

for entry_name, entry_col in entry_variants:
    n_entries = df_full[entry_col].sum()
    if n_entries == 0:
        print(f"\n{entry_name}: No entries - skipping")
        continue
        
    print(f"\n{'─'*110}")
    print(f"{entry_name} ({n_entries} entries)")
    print(f"{'─'*110}")
    print(f"{'Exit Strategy':<20} {'Return':>12} {'CAGR':>10} {'Sharpe':>10} {'Win%':>8} {'MaxDD':>10} {'Trades':>8} {'AvgHold':>8}")
    print("-" * 95)
    
    for exit_name, exit_type, exit_kwargs in exit_variants:
        if exit_type == 'trailing':
            trades = backtest_trailing_stop(df_full, entry_col)
        elif exit_type == 'onchain':
            trades = backtest_onchain_exit(df_full, entry_col, **exit_kwargs)
        elif exit_type == 'hybrid':
            trades = backtest_hybrid_exit(df_full, entry_col, **exit_kwargs)
        
        metrics = calc_metrics(trades)
        
        if metrics:
            print(f"{exit_name:<20} {metrics['total_return']:>+11.1%} {metrics['cagr']:>+9.1%} "
                  f"{metrics['sharpe']:>10.2f} {metrics['win_rate']:>7.0%} "
                  f"{metrics['max_drawdown']:>9.1%} {metrics['n_trades']:>8} {metrics['avg_hold']:>7.0f}d")
            
            all_results.append({
                'entry': entry_name,
                'exit': exit_name,
                'entry_col': entry_col,
                **metrics,
                'trades': trades,
            })
        else:
            print(f"{exit_name:<20} {'No trades':>12}")

In [ ]:
# Find best combinations
print("\n" + "=" * 90)
print("TOP 10 STRATEGIES (by Total Return)")
print("=" * 90)

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'trades'} for r in all_results])
results_df = results_df.sort_values('total_return', ascending=False)

print(f"\n{'Rank':<5} {'Entry':<20} {'Exit':<20} {'Return':>12} {'Sharpe':>10} {'Win%':>8} {'vs B&H':>12}")
print("-" * 95)

for i, (_, row) in enumerate(results_df.head(10).iterrows()):
    vs_bh = row['total_return'] - bh_return
    marker = "🏆" if i == 0 else ("🥈" if i == 1 else ("🥉" if i == 2 else "  "))
    beats = "✅" if vs_bh > 0 else "❌"
    print(f"{i+1:<5} {row['entry']:<20} {row['exit']:<20} {row['total_return']:>+11.1%} "
          f"{row['sharpe']:>10.2f} {row['win_rate']:>7.0%} {vs_bh:>+11.1%} {beats} {marker}")

print("-" * 95)
print(f"{'':5} {'Buy & Hold':<40} {bh_return:>+11.1%}")

In [ ]:
# Compare entry methods (average across exits)
print("\n" + "=" * 70)
print("ENTRY METHOD COMPARISON (averaged across exits)")
print("=" * 70)

entry_summary = results_df.groupby('entry').agg({
    'total_return': 'mean',
    'sharpe': 'mean',
    'win_rate': 'mean',
    'n_trades': 'mean',
}).sort_values('total_return', ascending=False)

print(f"\n{'Entry Method':<25} {'Avg Return':>12} {'Avg Sharpe':>12} {'Avg Win%':>10} {'Avg Trades':>12}")
print("-" * 75)
for entry, row in entry_summary.iterrows():
    print(f"{entry:<25} {row['total_return']:>+11.1%} {row['sharpe']:>12.2f} {row['win_rate']:>9.0%} {row['n_trades']:>12.1f}")

In [ ]:
# Compare exit methods (average across entries)
print("\n" + "=" * 70)
print("EXIT METHOD COMPARISON (averaged across entries)")
print("=" * 70)

exit_summary = results_df.groupby('exit').agg({
    'total_return': 'mean',
    'sharpe': 'mean',
    'win_rate': 'mean',
    'avg_hold': 'mean',
}).sort_values('total_return', ascending=False)

print(f"\n{'Exit Method':<25} {'Avg Return':>12} {'Avg Sharpe':>12} {'Avg Win%':>10} {'Avg Hold':>12}")
print("-" * 75)
for exit_m, row in exit_summary.iterrows():
    print(f"{exit_m:<25} {row['total_return']:>+11.1%} {row['sharpe']:>12.2f} {row['win_rate']:>9.0%} {row['avg_hold']:>11.0f}d")

## 6. Detailed Analysis of Best Strategy

In [ ]:
# Get best strategy
best = all_results[results_df.index[0]]

print(f"\n" + "=" * 90)
print(f"BEST STRATEGY: {best['entry']} + {best['exit']}")
print("=" * 90)

print(f"""
PERFORMANCE SUMMARY
{'─' * 50}
Total Return:     {best['total_return']:>+10.1%}
CAGR:             {best['cagr']:>+10.1%}
Sharpe Ratio:     {best['sharpe']:>10.2f}
Win Rate:         {best['win_rate']:>10.0%}
Max Drawdown:     {best['max_drawdown']:>10.1%}
Total Trades:     {best['n_trades']:>10}
Avg Hold (days):  {best['avg_hold']:>10.0f}

$100,000 → ${best['final_equity']:,.0f}
vs Buy & Hold: {best['total_return'] - bh_return:+.1%}
""")

# Trade list
trades = best['trades']
print("\nTRADE LIST:")
print("=" * 100)
print(f"{'#':>3} {'Entry':>12} {'Entry$':>10} {'Exit':>12} {'Exit$':>10} {'Peak$':>10} {'Return':>10} {'Days':>6} {'Reason':>12}")
print("-" * 100)

for i, row in trades.iterrows():
    sym = "✓" if row['net_return'] > 0 else "✗"
    print(f"{i+1:>3} {str(row['entry_date'].date()):>12} ${row['entry_price']:>8,.0f} "
          f"{str(row['exit_date'].date()):>12} ${row['exit_price']:>8,.0f} ${row['peak_price']:>8,.0f} "
          f"{row['net_return']:>+9.1%} {row['days_held']:>6} {row['exit_reason']:>12} {sym}")

## 7. Current Market Status

In [ ]:
latest = df_full.iloc[-1]

print("\n" + "=" * 80)
print(f"CURRENT MARKET STATUS ({latest.name.date()})")
print("=" * 80)

print(f"""
BTC Price: ${latest['price']:,.0f}

===== ENTRY CONDITIONS =====

Fixed Thresholds:
  1. STH-MVRV < 1.0:   {latest['mvrv_sth']:.4f}  {'✓' if latest['cond_mvrv_fixed'] else '✗'}
  2. STH-SOPR < 1.0:   {latest['sopr_sth']:.4f}  {'✓' if latest['cond_sopr_fixed'] else '✗'}
  3. RPLR < 1.0:       {latest['rplr']:.4f}  {'✓' if latest['cond_rplr_fixed'] else '✗'}
  4. Funding ≤ 0:      {latest['funding_rate']*100:.4f}%  {'✓' if latest['cond_funding_fixed'] else '✗'}
  5. Long Liq > Short: {latest['liq_ratio']:.2f}x  {'✓' if latest['cond_liq_fixed'] else '✗'}

Z-Score Confirmation (< -1.0 = unusually distressed):
  STH-MVRV Z:  {latest['mvrv_sth_z']:+.2f}  {'✓ UNUSUAL' if latest['mvrv_sth_z'] < -1.0 else '✗ normal'}
  STH-SOPR Z:  {latest['sopr_sth_z']:+.2f}  {'✓ UNUSUAL' if latest['sopr_sth_z'] < -1.0 else '✗ normal'}
  RPLR Z:      {latest['rplr_z']:+.2f}  {'✓ UNUSUAL' if latest['rplr_z'] < -1.0 else '✗ normal'}

Z-Confirms: {int(latest['z_confirm_count'])}/3

─────────────────────────────────────
ENTRY SIGNAL STATUS:
  Entry 1 (Fixed):     {'🟢 ACTIVE' if latest['signal_fixed'] else '⚫ inactive'}
  Entry 2 (Full Z):    {'🟢 ACTIVE' if latest['signal_z_confirmed'] else '⚫ inactive'}
  Entry 3 (Partial Z): {'🟢 ACTIVE' if latest['signal_partial_z'] else '⚫ inactive'}

===== EXIT CONDITIONS =====

Z-Score Euphoria Count: {int(latest['z_count_exit'])}/8
  Exit A (Mirror):     {'🔴 SELL' if latest['exit_a'] else '⚫ hold'}
  Exit B (6/8 Z):      {'🔴 SELL' if latest['exit_b'] else '⚫ hold'}
  Exit C (LTH Dist):   {'🔴 SELL' if latest['exit_c'] else '⚫ hold'}
""")

## 8. Summary & Recommendations

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

# Best entry
best_entry = entry_summary.index[0]
print(f"\n1. BEST ENTRY METHOD: {best_entry}")
print(f"   Avg Return: {entry_summary.loc[best_entry, 'total_return']:+.1%}")

# Best exit
best_exit = exit_summary.index[0]
print(f"\n2. BEST EXIT METHOD: {best_exit}")
print(f"   Avg Return: {exit_summary.loc[best_exit, 'total_return']:+.1%}")

# Best combo
print(f"\n3. BEST COMBINATION: {best['entry']} + {best['exit']}")
print(f"   Return: {best['total_return']:+.1%}")
print(f"   vs B&H: {best['total_return'] - bh_return:+.1%}")

# Z-score value
fixed_avg = entry_summary.loc['Entry 1: Fixed', 'total_return'] if 'Entry 1: Fixed' in entry_summary.index else 0
partial_z_avg = entry_summary.loc['Entry 3: Partial Z', 'total_return'] if 'Entry 3: Partial Z' in entry_summary.index else 0

print(f"\n4. Z-SCORE CONFIRMATION VALUE:")
print(f"   Fixed entry avg: {fixed_avg:+.1%}")
print(f"   Partial Z avg:   {partial_z_avg:+.1%}")
print(f"   Difference:      {partial_z_avg - fixed_avg:+.1%}")

if partial_z_avg > fixed_avg:
    print(f"   → Z-confirmation IMPROVES returns")
else:
    print(f"   → Z-confirmation REDUCES returns (filters too much)")